## 예제 2.1 토큰화 코드

In [1]:
# 띄어쓰기 단위로 분리
input_text = "나는 최근 파리 여행을 다녀왔다"
input_text_list = input_text.split()
print("input_text_list: ", input_text_list)

# 토큰 -> 아이디 딕셔너리와 아이디 -> 토큰 딕셔너리 만들기
str2idx = {word:idx for idx, word in enumerate(input_text_list)}
idx2str = {idx:word for idx, word in enumerate(input_text_list)}
print("str2idx: ", str2idx)
print("idx2str: ", idx2str)

# 토큰을 토큰 아이디로 변환
input_ids = [str2idx[word] for word in input_text_list]
print("input_ids: ", input_ids)

input_text_list:  ['나는', '최근', '파리', '여행을', '다녀왔다']
str2idx:  {'나는': 0, '최근': 1, '파리': 2, '여행을': 3, '다녀왔다': 4}
idx2str:  {0: '나는', 1: '최근', 2: '파리', 3: '여행을', 4: '다녀왔다'}
input_ids:  [0, 1, 2, 3, 4]


컴퓨터는 글자를 모릅니다. 숫자만 압니다. 글자를 숫자로 바꾸는 첫 단계입니다.
input_text.split() 공백 기준으로 자릅니다. 결과는 파이썬 리스트.

{word:idx for idx, word in enumerate(input_text_list)} 딕셔너리 컴프리헨션입니다. enumerate는 (0, '나는'), (1, '최근'), ... 형태로 인덱스와 값을 함께 돌려줍니다.

str2idx - 단어:번호 - 입력할 때 (인코딩)

idx2str - 번호:단어 - 출력할 때 (디코딩)

모델은 숫자를 뱉으므로, 사람이 읽으려면 idx2str로 되돌려야 합니다. 챗봇이 글자를 하나씩 뱉는 것처럼 보이는 것도 실은 숫자를 받아서 이 표로 되돌리는 과정입니다.

input_ids 최종 결과 [0, 1, 2, 3, 4]. 여기서는 단어가 처음 등장한 순서대로 번호를 매겼으므로 우연히 순서대로 나왔습니다.

Q. 어휘 사전을 문장 하나로 만들어도 되나요?

안됩니다. 이건 설명용 축소판입니다. 실제로는:
- 수십 GB의 텍스트 전체를 훑어서 사전을 만들고 사전의 크기는 보통 3만~15만개.
- 학습 전에 한번 만들어서 고정합니다.(문장마다 새로 만들지 않음)

Q. 사전에 없는 단어가 오면?

이 코드는 KeyError로 죽습니다. 실제 시스템은 [UNK](unknown)같은 특수 토큰을 준비해둡니다.

Q. 왜 띄어쓰기로 자르면 안되나요?

한국어에서 치명적입니다.

"여행을", "여행이", "여행은", "여행에서"

사람 눈에는 다 같은 "여행"인데, 이 방식에서는 전부 다른 번호를 받습니다. 모델은 이들이 관련 있다는 걸 처음부터 다시 배워야 합니다. 사전도 쓸데없이 커집니다.

그래서 실제 LLM은 서브워드(subword) 토크나이저를 씁니다.

"여행을" -> ["여행", "을"]

"여행이" -> ["여행", "이"]

"여행"이라는 조각을 공유하게 됩니다. 대표적으로 BPE(GPT), WordPiece(BERT), SentencePiece가 있고, 3장에서 허깅페이스 토크나이저로 실제 사례를 다룹니다.

In [2]:
print(list(enumerate(input_text_list)))

[(0, '나는'), (1, '최근'), (2, '파리'), (3, '여행을'), (4, '다녀왔다')]


## 예제 2.2 토큰 아이디에서 벡터로 변환

In [3]:
import torch
import torch.nn as nn

embedding_dim = 16
embed_layer = nn.Embedding(len(str2idx), embedding_dim)

input_embeddings = embed_layer(torch.tensor(input_ids)) # (5, 16)
input_embeddings = input_embeddings.unsqueeze(0) # (1, 5, 16)
input_embeddings.shape

torch.Size([1, 5, 16])

번호 하나(2)를 16개 숫자로 된 벡터로 바꿉니다. 이게 "임베딩"입니다.

왜 번호로는 부족한가?

파리 = 2, 여행을 = 3 이라고 할 때, 이 숫자를 그대로 모델에 넣으면 문제가 생깁니다.

- 2 < 3 이라는 대소 관계가 생김 -> 의미 없음
- 2 + 3 = 5 -> "파리 + 여행을 = 다녀왔다"? 말이 안됨
- 모든 단어가 1차원 직선 위에 놓임 -> 표현력이 거의 없음

벡터로 바꾸면 16차원 공간에 흩어놓을 수 있습니다. 학습이 끝나면 비슷한 단어끼리 가까이 모입니다.

nn.Embedding(len(str2idx), embedding_dim)

len(str2idx) - 5 - 사전 크기 = 표의 행 개수

embedding_dim - 16 - 임베팅 차원 = 표의 열 개수

즉, 5 X 16 크기의 표입니다. 처음에는 난수로 채워집니다.

embed_layer(torch.tensor(input_ids))

nn.Embedding의 동작은 단순합니다. 행을 뽑아오는 것뿐입니다.

input_ids = [0, 1, 2, 3, 4]

-> 표의 0행, 1행, 2행, 3행, 4행을 순서대로 꺼내 쌓음

-> (5, 16)

행렬곱이 아니라 인덱싱입니다. 사전이 10만 개짜리일 때 원핫 벡터를 만들어 곱하는 것보다 훨씬 빠릅니다.

.unsqueeze(0)
(5, 16) -> (1, 5, 16). 맨 앞에 크기 1짜리 차원을 추가합니다.

왜 필요한가?: PyTorch의 모든 층은 여러 문장을 한꺼번에 처리하도록 만들어져 있습니다. 문장이 하나뿐이어도 "1개짜리 배치"라고 알려줘야 합니다.

중요: 이 값들은 학습됩니다.

embed_layer.weight.requires_grad

지금은 난수지만 학습 대상 파라미터입니다. 역전파가 매 스텝 이 5 X 16 값을 갱신합니다.

GPT-3를 예로 들면 사전 5만개 X 차원 12,288 = 6억 개 파라미터가 임베딩 표 하나에 들어 있습니다.

초보자가 헷갈리는 지점

Q. embedding_dim = 16은 어떻게 정하나요?

하이퍼파라미터, 즉 사람이 정하는 값입니다. 실제 모델은 훨씬 큽니다.

BERT-base: 768차원

GPT-3: 12,288차원

크면 표현력이 좋아지지만 연산량과 메모리가 늘어납니다. 16은 눈으로 볼 수 있게 줄인 값입니다.

Q. 예제 2.2의 결과는 다음 셀에서 안쓰이나요?

네. 예제 2.3에서 embed_layer를 다시 만들어서 덮어씁니다. 2.2는 "토큰 임베딩만 있으면 이렇다"를 보여주는 단계이고, 2.3이 위치 인코딩까지 합친 완성본입니다.



In [4]:
input_embeddings

tensor([[[ 0.1284, -1.0152, -0.7475,  0.5164,  1.0550,  0.5055, -0.1030,
          -2.3228,  2.9543, -0.3333, -1.2496,  0.6335, -0.2080,  0.0034,
          -0.9777,  1.4815],
         [-0.7280, -1.5924,  0.0284, -0.4481,  0.7327, -0.8817, -0.2408,
          -1.1367, -0.1674,  0.6542, -0.8846,  0.5753,  0.9382,  1.0906,
          -0.4287,  2.0211],
         [ 1.6150, -1.2672, -0.6503,  0.3921,  0.0793,  0.0771,  0.6762,
           0.4942,  1.4332, -0.0729, -2.0592,  0.0141, -0.9701,  1.0298,
          -2.1497, -0.1452],
         [ 0.2234, -1.1372,  1.4814, -0.8688,  1.8252,  1.7724, -0.3963,
           0.8480, -0.4493,  1.3676,  1.1624,  1.3539,  0.9284,  0.8170,
          -0.1776,  0.2874],
         [ 0.2260,  0.5559,  0.0669, -1.1391, -0.0115, -0.3375,  1.3311,
          -0.0436, -0.4352, -0.9712, -1.1313, -0.7921, -0.7530, -0.4273,
           0.0383, -0.1722]]], grad_fn=<UnsqueezeBackward0>)

## 예제 2.3 절대적 위치 인코딩

In [5]:
embedding_dim = 16
max_position = 12
# 토큰 임베딩 층 생성
embed_layer = nn.Embedding(len(str2idx), embedding_dim)
# 위치 인코딩 층 생성
position_embed_layer = nn.Embedding(max_position, embedding_dim)

position_ids = torch.arange(len(input_ids), dtype=torch.long).unsqueeze(0)
position_encodings = position_embed_layer(position_ids)
token_embeddings = embed_layer(torch.tensor(input_ids)) # (5, 16)
token_embeddings = token_embeddings.unsqueeze(0) # (1, 5, 16)
# 토큰 임베딩과 위치 인코딩을 더해 최종 입력 임베딩 생성
input_embeddings = token_embeddings + position_encodings
input_embeddings.shape

torch.Size([1, 5, 16])

왜 위치 정보가 필요한가

셀프 어텐션은 모든 토큰을 동시에 봅니다. RNN처럼 앞에서부터 순서대로 읽지 않습니다.

빠른건 좋은데, 부작용이 있습니다. 순서 개념이 구조적으로 존재하지 않습니다.

"나는 파리를 좋아한다."

"파리는 나를 좋아한다."

위치 정보가 없으면 모델 입장에서 이 둘은 같은 토큰 집합일 뿐입니다. 구분할 방법이 없습니다.

max_potisition = 12

이 모델이 처리할 수 있는 최대 토큰 길이입니다. 13번째 토큰이 오면 IndexError가 납니다.

실제 모델에서 말하는 "컨텍스트 길이 8K/128K"가 바로 이 값입니다.

torch.arrange(len(input_ids), dtype=torch.long)

torch.arange(5)

range(5)의 텐서 버전입니다. "0번째, 1번째, ..., 4번째 자리"라는 위치 번호를 만듭니다.

dtype=torch.long - 임베딩 층의 입력은 반드시 정수여야 합니다. 인덱스니까요.

.unsqueeze(0)

(5, ) -> (1, 5). 토큰 임베딩과 모양을 맞추기 위한 배치 차원입니다.

potision_embed_layer(position_ids)

12 X 16 표에서 0~4행을 뽑아옵니다. -> (1, 5, 16)

token_embeddings + position_encodings

핵심입니다. concat이 아니라 덧셈입니다. 차원이 늘어나지 않고 (1, 5, 16)을 유지합니다.

덧셈이 하는 일 - 정확히 이해하기

input_embeddings[0][0][0] = token_embeddings[0][0][0] + position_encodings[0][0][0]

같은 위치의 값끼리 더하는 원소별 스칼라 덧셈입니다. 16개 칸이 각각 독립적으로 더해집니다.

같은 토큰이라도 위치가 다르면 다른 벡터가 되고, 그 차이가 출력에 영향을 줄 수 있는가

이건 만족합니다. 정보가 "복원 가능한 형태로" 남을 필요는 없고, "출력을 갈라놓을 수 있으면" 충분합니다.

한 가지 더 - 어텐션은 벡터 간의 관계(내적)로 동작합니다. 두 토큰을 비교할 때 공통 성분은 상쇄되는 경향이 있습니다.

그래도 대가는 있습니다.

간섭은 실제로 존재합니다. 위치 성분이 의미 성분을 오염시킵니다. 트랜스포머는 이런 손실을 여러 곳에서 감수하는 설계입니다.

그럼 왜 concat이 아니라 add인가? 비용 때문입니다.

concat 하면 768 -> 1536 차원이 되고, 어텐션 파라미터는 차원의 제곱에 비례하므로 4배로 늘어납니다. 층마다, 헤드마다 전부요. 그 대가로 사는 건 "위치 512개를 표현하려고 768차원을 통째로 쓰는 것"입니다. 낭비가 큽니다.

원 논문도 두 방식을 비교해보고 성능 차이가 거의 없다고 보고했습니다. 우아해서가 아니라, 싸게 해봤는데 되더라에 가깝습니다.



## 예제 2.4 쿼리, 키, 값 벡터를 만드는 nn.Linear 층

In [6]:
head_dim = 16

# 쿼리, 키, 값을 계산하기 위한 변환
weight_q = nn.Linear(embedding_dim, head_dim)
weight_k = nn.Linear(embedding_dim, head_dim)
weight_v = nn.Linear(embedding_dim, head_dim)
# 변환 수행
querys = weight_q(input_embeddings) # (1, 5, 16)
keys = weight_k(input_embeddings) # (1, 5, 16)
values = weight_v(input_embeddings) # (1, 5, 16)

In [17]:
querys

tensor([[[ 5.8976e-01, -7.5486e-01, -5.5848e-02,  4.4594e-01, -7.4737e-01,
           2.8177e-02,  7.5349e-02,  5.2093e-02,  1.9848e-01, -6.3409e-03,
          -5.7272e-02,  2.3548e-01, -1.0176e+00, -1.0819e-01, -1.3873e-01,
           1.6873e+00],
         [-3.8423e-01,  1.2114e+00, -5.8878e-01,  6.8149e-01,  3.8930e-01,
           1.5885e+00, -5.6479e-01,  9.5905e-04,  1.3159e+00, -1.7179e+00,
          -1.3639e+00, -1.1408e+00, -8.6870e-01, -2.7957e-03,  6.3405e-01,
           9.5551e-01],
         [-3.7607e-01, -1.2112e+00, -3.4599e-01,  5.7802e-01,  2.7274e-01,
           4.2716e-01, -2.3735e-01,  3.6042e-01, -1.4333e+00, -3.4405e-02,
           4.9881e-01,  3.5712e-02, -5.9578e-01,  1.3554e-01,  6.8498e-01,
          -1.0956e-01],
         [-9.1651e-01,  1.1741e+00, -9.5339e-01, -3.4070e-01, -1.5632e+00,
           3.1547e-01, -2.4119e-01, -2.0825e-02, -6.3160e-01, -1.0334e-01,
          -1.0240e-01,  1.4814e+00,  6.7493e-01, -1.2465e+00,  6.9659e-01,
          -4.8853e-01],
    

같은 입력을 세 가지 다른 관점으로 변환합니다.

Q/K/V 비유 - 도서관 검색

이게 어텐션에서 가장 중요한 개념입니다. 도서관에 비유하면:

Query(쿼리) - 내가 찾는 것 - 검색창에 친 검색어

Key(키) - 각 항목의 색인 - 책등에 붙은 제목/키워드

Value(값) - 실제 내용 - 책의 본문

동작 순서:
1. 검색어(Q)와 모든 책의 제목(K)을 비교해서 관련도 점수를 매김

2. 점수가 높은 책의 본문(V)을 많이 가져옴

3. 점수 비율대로 섞어서 결과를 만듦

문장에 적용하면:

"나는 최근 파리 여행을 다녀왔다"

"파리"라는 토큰의 Query가 뭍습니다.:

"나를 이해하려면 어떤 단어를 봐야 하지?"

각 토큰의 Key와 비교:

"여행을"의 Key -> 점수 높음 -> 아, 도시 파리구나

"다녀왔다"의 Key -> 점수 높음

"나는"의 Key -> 점수 낮음

-> "여행을", "다녀왔다"의 Value를 많이 섞어서 "파리"의 새로운 표현을 만듦

같은 "파리"라도 "파리가 날아다닌다"에서는 다른 토큰들과 관계 맺으므로 다른 벡터가 됩니다. 이게 문맥 반영입니다.

nn.Linear(16, 16)

선형 변환 층입니다. 내부에 16 X 16 가중치 행렬 W와 16개 편향 b를 가집니다.

출력 = 입력 @ W.T + b

nn.Embedding 과의 차이:

nn.Embedding - 표에서 행을 뽑음(인덱싱)

nn.Linear - 벡터에 행렬을 곱함(변환)

weight_q, weight_k, weight_v가 각각 따로 있는 이유

세 층은 모두 같은 input_embeddings를 입력받습니다. 그런데 가중치가 다르므로 결과가 달라집니다.

querys = weight_q(input_embeddings)

keys = weight_k(input_embeddings)

values = weight_v(input_embeddings)

만약 하나만 썼다면 Q = K = V가 되어 모든 토큰이 자기 자신과 가장 비슷하다고 판단하게 됩니다. 역할을 분리하려고 세 개로 나눈 것이빈다.

이 세 층의 가중치도 전부 학습 대상입니다. "무엇을 검색어로 삼을지, 무엇을 색인으로 삼을지"를 모델이 스스로 배웁니다.

셀프 어텐션이라는 이름

Q, K, V가 전부 같은 입력에서 나왔습니다. 자기 자신을 검색하는 셈이라 "셀프(self) 어텐션"입니다.

예제 2.14의 크로스 어텐션에서는 Q는 디코더에서, K와 V는 인코더에서 옵니다. 그때 대비가 명확해집니다.



## 예제 2.5. 스케일 점곱 방식의 어텐션

In [7]:
from math import sqrt
import torch.nn.functional as F

def compute_attention(querys, keys, values, is_causal=False):
	dim_k = querys.size(-1) # 16
	scores = querys @ keys.transpose(-2, -1) / sqrt(dim_k)
	weights = F.softmax(scores, dim=-1)
	return weights @ values

In [8]:
querys

tensor([[[ 5.8976e-01, -7.5486e-01, -5.5848e-02,  4.4594e-01, -7.4737e-01,
           2.8177e-02,  7.5349e-02,  5.2093e-02,  1.9848e-01, -6.3409e-03,
          -5.7272e-02,  2.3548e-01, -1.0176e+00, -1.0819e-01, -1.3873e-01,
           1.6873e+00],
         [-3.8423e-01,  1.2114e+00, -5.8878e-01,  6.8149e-01,  3.8930e-01,
           1.5885e+00, -5.6479e-01,  9.5905e-04,  1.3159e+00, -1.7179e+00,
          -1.3639e+00, -1.1408e+00, -8.6870e-01, -2.7957e-03,  6.3405e-01,
           9.5551e-01],
         [-3.7607e-01, -1.2112e+00, -3.4599e-01,  5.7802e-01,  2.7274e-01,
           4.2716e-01, -2.3735e-01,  3.6042e-01, -1.4333e+00, -3.4405e-02,
           4.9881e-01,  3.5712e-02, -5.9578e-01,  1.3554e-01,  6.8498e-01,
          -1.0956e-01],
         [-9.1651e-01,  1.1741e+00, -9.5339e-01, -3.4070e-01, -1.5632e+00,
           3.1547e-01, -2.4119e-01, -2.0825e-02, -6.3160e-01, -1.0334e-01,
          -1.0240e-01,  1.4814e+00,  6.7493e-01, -1.2465e+00,  6.9659e-01,
          -4.8853e-01],
    

트랜스포머 전체에서 가장 중요한 4줄입니다. 논문 제목이 "Attention is All You Need"인 이유가 여기 있습니다.

1행: dim_k = querys.size(-1)

마지막 차원의 크기를 가져옵니다. (1, 5, 16)이면 16.

-1은 파이썬의 "맨 뒤" 인덱싱입니다. 3차원이든 4차원이든 항상 마지막을 가리키므로, 예제 2.8의 4차원 텐서에서도 그대로 동작합니다.

2행: querys @ keys.transpose(-2, -1)

@는 행렬곱 연산자입니다.

.transpose(-2, -1)은 뒤에서 2번째와 1번째 차원을 바꿉니다.

keys: (1, 5, 16)

keys.transpose(-2, -1): (1, 16, 5)

왜 바꿔야 하는가 - 행렬곱은 앞의 열 개수와 뒤의 행 개수가 같아야 합니다.

(1, 5, 16) @ (1, 16, 5) -> (1, 5, 5)

결과 (1, 5, 5)의 의미

5 X 5표가 나옵니다. 모든 토큰 쌍의 관련도 점수표입니다.

[2][3] 칸 = "파리"의 Queyr와 "여행을"의 Key를 내적한 값 = 두 토큰의 관련도.

내적이 클수록 두 벡터가 비슷한 방향입니다. (5, 16) @ (16, 5) 한번으로 25개 쌍의 관련도가 전부 계산됩니다. 이 병렬성이 트랜스포머가 RNN보다 빠른 이유입니다.

2행 뒷부분: /sqrt(dim_k)

"스케일(Scaled)" 점곱이라는 이름이 여기서 나옵니다.

왜 나누는가:

16개 항을 더해서 만든 내적은 차원이 클수록 값이 커집니다. 통계적으로 내적의 표준 편차는 /sqrt(dim_k)에 비례합니다.

점수가 너무 크면 다음 단계인 softmax가 포화됩니다.

softmax([1, 2, 3]) -> [0.09, 0.24, 0.67] 좋음
softmax([10, 20, 30]) -> [0.00, 0.00, 1.00] 한 곳에 몰림

한 곳에 1.0이 몰리면 나머지 토큰의 그레이디언트가 거의 0이 되어 학습이 멈춥니다. /sqrt(d_k)로 나눠서 점수 분포를 적당한 범위로 되돌리는 겁니다.

F.softmax(scores, dim=-1)

점수를 확률로 바꿉니다. 모든 값이 0~1이 되고, 합이 정확히 1이 됩니다.

"파리"행: [0.01, 0.02, 0.55, 0.28, 0.14] 합 = 1.0

읽는 법: "파리를 이해할 때 자기 자신에 55%, '여행을'에 28%, '다녀왔다'에 14% 주의를 준다."

만약 dim = 0 이나 dim = 1로 잘못쓰면 열 방향으로 정규화되어 의미가 완전히 달라집니다. 어텐션 구현에서 가장 흔한 실수 중 하나입니다.

-1은 마지막 차원을 의미합니다.

weights @ values

(1, 5, 5) @ (1, 5, 16) -> (1, 5, 16)

가중치대로 Value를 섞습니다.




## 예제 2.6. 어텐션 연산의 입력과 출력

In [18]:
print("원본 입력 형태: ", input_embeddings.shape)

after_attention_embeddings = compute_attention(querys, keys, values)

print("어텐션 적용 후 형태: ", after_attention_embeddings.shape)
# 원본 입력 형태:  torch.Size([1, 5, 16])
# 어텐션 적용 후 형태:  torch.Size([1, 5, 16])

원본 입력 형태:  torch.Size([1, 5, 16])
어텐션 적용 후 형태:  torch.Size([1, 5, 16])


어텐션은 텐서 모양을 바꾸지 않습니다. 이게 트랜스포머 설계의 핵심입니다.

값은 완전히 바뀌었지만 모양은 같습니다. 그래서:

1. 어텐션 블록을 몇 개든 이어붙일 수 있습니다. (출력이 다음 입력이 됨)

2. 잔차 연결(x + f(x))이 가능합니다. - 모양이 달랐다면 할 수 없습니다.

GPT-3가 96층을 쌓을 수 있었던 것도 이 성질 덕분입니다. 층을 늘리는 게 그냥 for 문 반복이 됩니다.(예제 2.12에서 실제로 그렇게 합니다.)


## 예제 2.7. 어텐션 연산을 수행하는 AttentionHead 클래스

In [ ]:
class AttentionHead(nn.Module):
  def __init__(self, token_embed_dim, head_dim, is_causal=False):
    super().__init__()
    self.is_causal = is_causal
    self.weight_q = nn.Linear(token_embed_dim, head_dim) # 쿼리 벡터 생성을 위한 선형 층
    self.weight_k = nn.Linear(token_embed_dim, head_dim) # 키 벡터 생성을 위한 선형 층
    self.weight_v = nn.Linear(token_embed_dim, head_dim) # 값 벡터 생성을 위한 선형 층

  def forward(self, querys, keys, values):
    outputs = compute_attention(
        self.weight_q(querys),  # 쿼리 벡터
        self.weight_k(keys),    # 키 벡터
        self.weight_v(values),  # 값 벡터
        is_causal=self.is_causal
    )
    return outputs

attention_head = AttentionHead(embedding_dim, embedding_dim)
after_attention_embeddings = attention_head(input_embeddings, input_embeddings, input_embeddings)

예제 2.4 + 2.5를 하나의 재사용 가능한 부품으로 포장합니다. 새로운 연산은 없습니다.

nn.Module 기초

PyTorch에서 모든 신경망 구성요소는 nn.Module을 상속합니다.

상속하면 얻는 것:
- .parameters() - 내부의 모든 학습 파라미터를 자동 수집(옵티마이저에 넘길 때 필수)

- .to('cuba') - 모든 파라미터를 GPU로 한번에 이동

- .train() / .eval() - 드롭아웃 등의 동작 모드 전환

- .state_dict() - 저장 / 불러오기

super().__init__()

부모 클래스의 초기화를 먼저 호출합니다. 이걸 빼먹으면 파라미터 등록이 안 되어 학습이 되지 않습니다. 가장 흔한 초보 실수입니다.

self.weight_q = nn.Linear(...)

self. 에 할당하는 순간 Py Torch가 자동으로 추천 목록에 넣습니다. 그냥 지역 변수 weight_q = ... 로 쓰면 등록되지 않습니다.

forward 매서드

attention_head(input_embeddings, ...) # 이렇게 호출하면

attention_head.forward(...) # 자동으로 이게 실행됨

forward를 직접 부르지 않는게 관례입니다. __call__이 forward 전후로 hook 처리 등을 해주기 때문입니다.

인자 이름에 대한 주의

forward(self, querys, keys, values) 라고 되어 있지만, 여기 들어오는 건 아직 Q/K/V가 아닙니다. 그냥 입력 임베딩입니다.

attention_head(input_embeddings, input_embeddings, input_embeddings)

내부에서 self.weight_q(querys)를 거쳐야 비로소 진짜 Query가 됩니다. 이름이 헷갈리게 붙어 있으니 "Q로 변환된 재로"정도로 읽으면 됩니다.

왜 인자를 세 개나 받나

셋 다 같은 값을 넣을 거면 하나면 될 것 같지만, 크로스 어텐션 때문입니다.

인터페이스를 세 개로 열어두면 두 경우를 같은 클래스로 처리할 수 있습니다.

## 예제 2.8. 멀티 헤드 어텐션 구현

In [ ]:
class MultiheadAttention(nn.Module):
  def __init__(self, token_embed_dim, d_model, n_head, is_causal=False):
    super().__init__()
    self.n_head = n_head
    self.is_causal = is_causal
    self.weight_q = nn.Linear(token_embed_dim, d_model)
    self.weight_k = nn.Linear(token_embed_dim, d_model)
    self.weight_v = nn.Linear(token_embed_dim, d_model)
    self.concat_linear = nn.Linear(d_model, d_model)

  def forward(self, querys, keys, values):
    B, T, C = querys.size()
    querys = self.weight_q(querys).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
    keys = self.weight_k(keys).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
    values = self.weight_v(values).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
    attention = compute_attention(querys, keys, values, self.is_causal)
    output = attention.transpose(1, 2).contiguous().view(B, T, C)
    output = self.concat_linear(output)
    return output

n_head = 4
mh_attention = MultiheadAttention(embedding_dim, embedding_dim, n_head)
after_attention_embeddings = mh_attention(input_embeddings, input_embeddings, input_embeddings)
after_attention_embeddings.shape

헤드가 하나면 어텐션 패턴도 하나뿐입니다. 그런데 문장에는 여러 종류의 관계가 동시에 존재합니다.

"나는 최근 파리 여행을 다녀왔다."

문법 관계: "나는" <-> "다녀왔다" (주어 - 서술어)

의미 관계: "파리" <-> "여행을" (장소 - 행위)

시간 관계: "최근" <-> "다녀왔다" (시점 - 동작)

하나의 어텐션으로 이걸 다 잡으려면 무리입니다. 여러 개를 병렬로 돌려서 각자 다른 관계를 학습하게 하는 것이 멀티 헤드입니다.

핵심 아이디어: 나눠서 병렬 처리

16차원 -> 4개 헤드 X 4차원씩

차원을 늘리는 게 아니라 쪼갭니다. 그래서 헤드를 늘려도 연산량이 거의 그대로입니다.

d_model = 16, n_head = 4 -> head_dim = 16 / 4 = 4

헤드 0: 0~3번 차원 담당

헤드 1: 4~7번 차원 담당

헤드 2: 8~11번 차원 담당

헤드 3: 12~15번 차원 담당

B, T, C = querys.size()

파이썬 언패킹입니다. (1, 5, 16)이 각각 배치/토큰수/차원으로 풀립니다.

이 세 글자는 트랜스포머 코드의 관례이고, nanoGPR 같은 유명 구현에서도 그대로 씁니다.

.view(B, T, self.n_head, C // self.n_head)

가장 어려운 부분입니다. 천천히 봅시다.



## 예제 2.9. 층 정규화 코드

In [ ]:
norm = nn.LayerNorm(embedding_dim)
norm_x = norm(input_embeddings)
norm_x.shape # torch.Size([1, 5, 16])

norm_x.mean(dim=-1).data, norm_x.std(dim=-1).data

# (tensor([[ 2.2352e-08, -1.1176e-08, -7.4506e-09, -3.9116e-08, -1.8626e-08]]),
#  tensor([[1.0328, 1.0328, 1.0328, 1.0328, 1.0328]]))

## 예제 2.10. 피드 포워드 층 코드

In [ ]:
class PreLayerNormFeedForward(nn.Module):
  def __init__(self, d_model, dim_feedforward, dropout):
    super().__init__()
    self.linear1 = nn.Linear(d_model, dim_feedforward) # 선형 층 1
    self.linear2 = nn.Linear(dim_feedforward, d_model) # 선형 층 2
    self.dropout1 = nn.Dropout(dropout) # 드랍아웃 층 1
    self.dropout2 = nn.Dropout(dropout) # 드랍아웃 층 2
    self.activation = nn.GELU() # 활성 함수
    self.norm = nn.LayerNorm(d_model) # 층 정규화

  def forward(self, src):
    x = self.norm(src)
    x = x + self.linear2(self.dropout1(self.activation(self.linear1(x))))
    x = self.dropout2(x)
    return x

## 예제 2.11. 인코더 층

In [ ]:
class TransformerEncoderLayer(nn.Module):
  def __init__(self, d_model, nhead, dim_feedforward, dropout):
    super().__init__()
    self.attn = MultiheadAttention(d_model, d_model, nhead) # 멀티 헤드 어텐션 클래스
    self.norm1 = nn.LayerNorm(d_model) # 층 정규화
    self.dropout1 = nn.Dropout(dropout) # 드랍아웃
    self.feed_forward = PreLayerNormFeedForward(d_model, dim_feedforward, dropout) # 피드포워드

  def forward(self, src):
    norm_x = self.norm1(src)
    attn_output = self.attn(norm_x, norm_x, norm_x)
    x = src + self.dropout1(attn_output) # 잔차 연결

    # 피드 포워드
    x = self.feed_forward(x)
    return x

## 예제 2.12. 인코더 구현

In [ ]:
import copy
def get_clones(module, N):
  return nn.ModuleList([copy.deepcopy(module) for i in range(N)])

class TransformerEncoder(nn.Module):
  def __init__(self, encoder_layer, num_layers):
    super().__init__()
    self.layers = get_clones(encoder_layer, num_layers)
    self.num_layers = num_layers
    self.norm = norm

  def forward(self, src):
    output = src
    for mod in self.layers:
        output = mod(output)
    return output

## 예제 2.13. 디코더에서 어텐션 연산(마스크 어텐션)

In [ ]:
def compute_attention(querys, keys, values, is_causal=False):
	dim_k = querys.size(-1) # 16
	scores = querys @ keys.transpose(-2, -1) / sqrt(dim_k) # (1, 5, 5)
	if is_causal:
		query_length = querys.size(2)
		key_length = keys.size(2)
		temp_mask = torch.ones(query_length, key_length, dtype=torch.bool).tril(diagonal=0)
		scores = scores.masked_fill(temp_mask == False, float("-inf"))
	weights = F.softmax(scores, dim=-1) # (1, 5, 5)
	return weights @ values # (1, 5, 16)

## 예제 2.14. 크로스 어텐션이 포함된 디코더 층

In [ ]:
class TransformerDecoderLayer(nn.Module):
  def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
    super().__init__()
    self.self_attn = MultiheadAttention(d_model, d_model, nhead)
    self.multihead_attn = MultiheadAttention(d_model, d_model, nhead)
    self.feed_forward = PreLayerNormFeedForward(d_model, dim_feedforward, dropout)

    self.norm1 = nn.LayerNorm(d_model)
    self.norm2 = nn.LayerNorm(d_model)
    self.dropout1 = nn.Dropout(dropout)
    self.dropout2 = nn.Dropout(dropout)

  def forward(self, tgt, encoder_output, is_causal=True):
    # 셀프 어텐션 연산
    x = self.norm1(tgt)
    x = x + self.dropout1(self.self_attn(x, x, x, is_causal=is_causal))
    # 크로스 어텐션 연산
    x = self.norm2(x)
    x = x + self.dropout2(self.multihead_attn(x, encoder_output, encoder_output))
    # 피드 포워드 연산
    x = self.feed_forward(x)
    return x

## 예제 2.15. 디코더 구현

In [ ]:
import copy
def get_clones(module, N):
  return nn.ModuleList([copy.deepcopy(module) for i in range(N)])

class TransformerDecoder(nn.Module):
  def __init__(self, decoder_layer, num_layers):
    super().__init__()
    self.layers = get_clones(decoder_layer, num_layers)
    self.num_layers = num_layers

  def forward(self, tgt, src):
    output = tgt
    for mod in self.layers:
        output = mod(output, src)
    return output